# Project Risk Prediction - ML Experiment

**Author:** Nada Ramadan  
**Dataset:** Project Management Risk Dataset (Kaggle)  
**Objective:** Predict project risk levels (Low, Medium, High, Critical)

---

## 1. Import Libraries

In [ ]:
# Data manipulation and analysis
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# File and system operations
import os
import joblib

# Machine Learning - Preprocessing
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV

# Machine Learning - Models
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Handling imbalanced data
from imblearn.over_sampling import SMOTE

# Advanced models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

print("All libraries imported successfully!")

All libraries imported successfully!


## 2. Load Dataset

In [ ]:
# Load the dataset
data = pd.read_csv('/content/project_risk_raw_dataset.csv')
print(f"Dataset loaded: {data.shape[0]} rows, {data.shape[1]} columns")

Dataset loaded: 4000 rows, 51 columns


In [ ]:
# Display first few rows
data.head(10)

,Project_ID,Project_Type,Team_Size,Project_Budget_USD,Estimated_Timeline_Months,Complexity_Score,Stakeholder_Count,Methodology_Used,Team_Experience_Level,Past_Similar_Projects,...,Industry_Volatility,Client_Experience_Level,Change_Control_Maturity,Risk_Management_Maturity,Team_Colocation,Documentation_Quality,Project_Start_Month,Current_Phase_Duration_Months,Seasonal_Risk_Factor,Risk_Level
0,PROJ_0001,Construction,32,1526276.55,32,9.70,16,Waterfall,Senior,3,...,Extreme,First-time,Basic,Basic,Fully Colocated,Good,10,5,1.0,High
1,PROJ_0002,Manufacturing,2,390790.15,9,2.72,9,Kanban,Mixed,0,...,Stable,Occasional,Advanced,Formal,Fully Remote,Poor,9,3,1.0,Low
2,PROJ_0003,Manufacturing,2,246674.76,6,2.04,7,Agile,Mixed,1,...,Stable,Regular,NaN,NaN,Hybrid,Good,5,1,1.0,Medium
3,PROJ_0004,IT,12,1427830.63,17,7.54,16,Scrum,Mixed,0,...,Extreme,Strategic,Formal,Basic,Hybrid,Basic,12,6,1.1,High
4,PROJ_0005,Construction,24,1696746.64,24,6.68,17,Hybrid,Junior,0,...,Moderate,Occasional,Basic,NaN,Partially Colocated,Basic,9,6,1.0,High
5,PROJ_0006,IT,13,1106456.85,18,7.11,12,Kanban,Mixed,0,...,Stable,Strategic,Formal,Advanced,Hybrid,Basic,3,7,1.0,Medium
6,PROJ_0007,R&D,12,1122048.59,21,10.00,11,Scrum,Senior,3,...,High,First-time,Basic,NaN,Fully Colocated,Good,9,7,1.0,Critical
7,PROJ_0008,Healthcare,9,1208078.86,15,4.53,9,Kanban,Mixed,3,...,High,Strategic,Formal,Basic,Partially Colocated,Poor,11,2,1.0,Low
8,PROJ_0009,Construction,37,1833354.63,35,9.49,13,Waterfall,Mixed,0,...,High,Occasional,Basic,Formal,Fully Colocated,Basic,1,5,1.0,Critical
9,PROJ_0010,Construction,41,2404494.18,28,5.09,18,Waterfall,Senior,2,...,Moderate,Strategic,NaN,NaN,Fully Colocated,Basic,2,9,1.0,Medium


## 3. Exploratory Data Analysis (EDA)

In [ ]:
data.shape

(4000, 51)

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 51 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Project_ID                       4000 non-null   object 
 1   Project_Type                     4000 non-null   object 
 2   Team_Size                        4000 non-null   int64  
 3   Project_Budget_USD               4000 non-null   float64
 4   Estimated_Timeline_Months        4000 non-null   int64  
 5   Complexity_Score                 4000 non-null   float64
 6   Stakeholder_Count                4000 non-null   int64  
 7   Methodology_Used                 4000 non-null   object 
 8   Team_Experience_Level            4000 non-null   object 
 9   Past_Similar_Projects            4000 non-null   int64  
 10  External_Dependencies_Count      4000 non-null   int64  
 11  Change_Request_Frequency         4000 non-null   float64
 12  Project_Phase       

In [ ]:
data.describe()

,Team_Size,Project_Budget_USD,Estimated_Timeline_Months,Complexity_Score,Stakeholder_Count,Past_Similar_Projects,External_Dependencies_Count,Change_Request_Frequency,Team_Turnover_Rate,Vendor_Reliability_Score,...,Market_Volatility,Integration_Complexity,Resource_Availability,Organizational_Change_Frequency,Cross_Functional_Dependencies,Previous_Delivery_Success_Rate,Technical_Debt_Level,Project_Start_Month,Current_Phase_Duration_Months,Seasonal_Risk_Factor
count,4000.000000,4.000000e+03,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,...,4000.00000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.00000,4000.000000,4000.000000,4000.000000
mean,15.388250,1.143032e+06,17.147750,6.192525,11.130500,1.973750,3.127750,1.638080,0.292725,0.712087,...,0.49716,5.573585,0.651695,1.212215,3.549250,0.750437,0.17544,6.492500,4.074500,1.011325
std,9.220969,5.908781e+05,6.926609,2.212538,4.425875,1.750093,1.609216,1.170451,0.166546,0.163418,...,0.28702,2.606654,0.201163,0.969191,2.320004,0.143712,0.29682,3.476416,2.883926,0.031694
min,2.000000,1.593555e+05,2.000000,1.620000,2.000000,0.000000,0.000000,0.010000,0.000000,0.090000,...,0.00000,1.000000,0.300000,0.000000,0.000000,0.150000,0.00000,1.000000,1.000000,1.000000
25%,9.000000,6.925329e+05,12.000000,4.460000,8.000000,1.000000,2.000000,0.760000,0.160000,0.600000,...,0.25000,3.317500,0.480000,0.500000,2.000000,0.660000,0.00000,4.000000,2.000000,1.000000
50%,13.000000,1.007472e+06,17.000000,6.015000,10.000000,2.000000,3.000000,1.370000,0.270000,0.730000,...,0.50000,5.600000,0.650000,0.960000,4.000000,0.770000,0.00000,7.000000,3.000000,1.000000
75%,20.000000,1.475870e+06,22.000000,7.862500,14.000000,3.000000,4.000000,2.230000,0.400000,0.840000,...,0.74000,7.860000,0.820000,1.660000,6.000000,0.860000,0.28000,10.000000,6.000000,1.000000
max,50.000000,3.768354e+06,36.000000,10.000000,29.000000,10.000000,7.000000,8.840000,0.850000,1.000000,...,1.00000,10.000000,1.000000,8.230000,7.000000,0.990000,1.00000,12.000000,17.000000,1.100000


## 4. Data Preprocessing

Handle missing values, drop unnecessary columns, and prepare data for modeling.

In [ ]:
data.isnull().sum()

,0
Project_ID,0
Project_Type,0
Team_Size,0
Project_Budget_USD,0
Estimated_Timeline_Months,0
Complexity_Score,0
Stakeholder_Count,0
Methodology_Used,0
Team_Experience_Level,0
Past_Similar_Projects,0


In [ ]:
data.drop('Project_ID', axis=1, inplace=True)

In [ ]:
# Drop 'Tech_Environment_Stability' column if it exists
if 'Tech_Environment_Stability' in data.columns:
    data.drop('Tech_Environment_Stability', axis=1, inplace=True)

# Impute missing values in 'Change_Control_Maturity' and 'Risk_Management_Maturity' with 'Unknown'
data['Change_Control_Maturity'] = data['Change_Control_Maturity'].fillna('Unknown')
data['Risk_Management_Maturity'] = data['Risk_Management_Maturity'].fillna('Unknown')

print("Missing values after cleaning:")
print(data[['Change_Control_Maturity', 'Risk_Management_Maturity']].isnull().sum())

Missing values after cleaning:
Change_Control_Maturity     0
Risk_Management_Maturity    0
dtype: int64


In [ ]:
# Separate features and target
X = data.drop('Risk_Level', axis=1)
y = data['Risk_Level']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

Features shape: (4000, 48)
Target shape: (4000,)


## 5. Feature Engineering

Create new features to capture important relationships and interactions.

In [ ]:
X['Budget_per_TeamMember'] = data['Project_Budget_USD'] / data['Team_Size']
X['Timeline_per_TeamMember'] = data['Estimated_Timeline_Months'] / data['Team_Size']
X['Risk_Index'] = (
    data['Historical_Risk_Incidents'] +
    data['Change_Request_Frequency'] +
    data['Team_Turnover_Rate']
)
X['Dependency_Load'] = (
    data['External_Dependencies_Count'] +
    data['Cross_Functional_Dependencies']
)
# Update numerical features list after adding engineered ones
num_features = X.select_dtypes(include=['int64','float64']).columns.tolist()

In [ ]:
# Focused interaction features from top 15 important features

# 1. Complexity × Team Turnover
X['Complexity_Turnover'] = X['Complexity_Score'] * X['Team_Turnover_Rate']

# 2. Complexity × Timeline
X['Complexity_Timeline'] = X['Complexity_Score'] * X['Estimated_Timeline_Months']

# 3. Budget ÷ Timeline
X['Budget_Timeline_Ratio'] = X['Project_Budget_USD'] / (X['Estimated_Timeline_Months'] + 1e-6)

# 4. Budget ÷ Team Member
X['Budget_TeamMember_Ratio'] = X['Project_Budget_USD'] / (X['Budget_per_TeamMember'] + 1e-6)

# 5. Timeline ÷ Team Member
X['Timeline_TeamMember_Ratio'] = X['Estimated_Timeline_Months'] / (X['Timeline_per_TeamMember'] + 1e-6)

# 6. Dependencies × Timeline
X['Dependencies_Timeline'] = X['External_Dependencies_Count'] * X['Estimated_Timeline_Months']

# 7. Communication × Change Requests
X['Comm_Change_Interaction'] = X['Communication_Frequency'] * X['Change_Request_Frequency']

# 8. Market Volatility × Integration Complexity
X['Market_Integration'] = X['Market_Volatility'] * X['Integration_Complexity']

# 9. Resource Availability ÷ Organizational Change
X['Resource_Change_Ratio'] = X['Resource_Availability'] / (X['Organizational_Change_Frequency'] + 1e-6)

# 10. Previous Delivery Success × Complexity
X['Success_Complexity'] = X['Previous_Delivery_Success_Rate'] * X['Complexity_Score']

# 11. Budget Utilization × Budget
X['Budget_Utilization_Impact'] = X['Budget_Utilization_Rate'] * X['Project_Budget_USD']

# 12. Timeline ÷ Change Requests
X['Timeline_Change_Ratio'] = X['Estimated_Timeline_Months'] / (X['Change_Request_Frequency'] + 1e-6)

# 13. Team Turnover ÷ Resource Availability
X['Turnover_Resource_Ratio'] = X['Team_Turnover_Rate'] / (X['Resource_Availability'] + 1e-6)

# 14. Dependencies ÷ Communication
X['Dependencies_Comm_Ratio'] = X['External_Dependencies_Count'] / (X['Communication_Frequency'] + 1e-6)

# 15. Timeline ÷ Market Volatility
X['Timeline_Market_Ratio'] = X['Estimated_Timeline_Months'] / (X['Market_Volatility'] + 1e-6)


## 6. Train/Test Split

Separate features and target, then split into training and testing sets.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib

from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [ ]:
X['Budget_per_TeamMember'] = data['Project_Budget_USD'] / data['Team_Size']
X['Timeline_per_TeamMember'] = data['Estimated_Timeline_Months'] / data['Team_Size']
X['Risk_Index'] = (
    data['Historical_Risk_Incidents'] +
    data['Change_Request_Frequency'] +
    data['Team_Turnover_Rate']
)
X['Dependency_Load'] = (
    data['External_Dependencies_Count'] +
    data['Cross_Functional_Dependencies']
)
# Update numerical features list after adding engineered ones
num_features = X.select_dtypes(include=['int64','float64']).columns.tolist()

In [ ]:
from sklearn.model_selection import train_test_split

X = data.drop(columns=['Risk_Level'])
y = data['Risk_Level']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")

Training set size: (3200, 48)
Testing set size: (800, 48)


In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

# Apply preprocessing to X to get transformed features
X_transformed = preprocessor.fit_transform(X)

X_resampled, y_resampled = SMOTE(random_state=42).fit_resample(
    X_transformed,
    y
)

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42
)

ValueError: Found array with 0 feature(s) (shape=(4000, 0)) while a minimum of 1 is required by SMOTE.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print("Random Forest with SMOTE:\n", classification_report(y_test, y_pred_rf))


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Use engineered + scaled + encoded features directly (no PCA)
X_features = preprocessor.fit_transform(X)
y_target = y

# Apply SMOTE for balance
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_features, y_target)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42
)

# Random Forest with tuned parameters
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print("Random Forest (with SMOTE):\n", classification_report(y_test, y_pred))


In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Apply SMOTE using the preprocessed X_transformed data
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_transformed, y)

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42
)

# Encode target labels to numerical values
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=12,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# Fit the model with the encoded target variables
xgb.fit(X_train, y_train_encoded)
y_pred_xgb_encoded = xgb.predict(X_test)

# Decode predictions back to original labels for classification report
y_pred_xgb = le.inverse_transform(y_pred_xgb_encoded)

print("XGBoost Results:\n", classification_report(y_test, y_pred_xgb))

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder # Import LabelEncoder

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)

# Encode target labels to numerical values for XGBoost
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)


xgb_model = XGBClassifier(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=7,
    scale_pos_weight=9 # Handle imbalance
)

# Fit the model with the encoded target variables
xgb.fit(X_train, y_train_encoded)
y_pred_xgb_encoded = xgb.predict(X_test)

# Decode predictions back to original labels for classification report
y_pred_xgb = le.inverse_transform(y_pred_xgb_encoded)

print("XGBoost Results:\n", classification_report(y_test, y_pred_xgb))

In [ ]:
from imblearn.over_sampling import SMOTE
import pandas as pd # Import pandas for Series.value_counts()

X_transformed = preprocessor.fit_transform(X)
smote = SMOTE(random_state=42) # Using SMOTE instead of ADASYN
X_resampled, y_resampled = smote.fit_resample(X_transformed, y_encoded)

print("Resampled class distribution:\n", pd.Series(y_resampled).value_counts())

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print("Random Forest with SMOTE:\n", classification_report(y_test, y_pred_rf))


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Use engineered + scaled + encoded features directly (no PCA)
X_features = preprocessor.fit_transform(X)  # preprocessor from earlier steps
y_target = y

# Apply SMOTE for balance
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_features, y_target)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42
)

# Random Forest with tuned parameters
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print("Random Forest (with SMOTE):\n", classification_report(y_test, y_pred))


In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Apply SMOTE using the preprocessed X_transformed data
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_transformed, y)

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42
)

# Encode target labels to numerical values
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=12,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# Fit the model with the encoded target variables
xgb.fit(X_train, y_train_encoded)
y_pred_xgb_encoded = xgb.predict(X_test)

# Decode predictions back to original labels for classification report
y_pred_xgb = le.inverse_transform(y_pred_xgb_encoded)

print("XGBoost Results:\n", classification_report(y_test, y_pred_xgb))

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder # Import LabelEncoder

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)

# Encode target labels to numerical values for XGBoost
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)


xgb_model = XGBClassifier(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=7,
    scale_pos_weight=9 # Handle imbalance
)

# Fit the model with the encoded target variables
xgb.fit(X_train, y_train_encoded)
y_pred_xgb_encoded = xgb.predict(X_test)

# Decode predictions back to original labels for classification report
y_pred_xgb = le.inverse_transform(y_pred_xgb_encoded)

print("XGBoost Results:\n", classification_report(y_test, y_pred_xgb))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print("Random Forest with SMOTE:\n", classification_report(y_test, y_pred_rf))


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Use engineered + scaled + encoded features directly (no PCA)
X_features = preprocessor.fit_transform(X)  # preprocessor from earlier steps
y_target = y

# Apply SMOTE for balance
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_features, y_target)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42
)

# Random Forest with tuned parameters
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print("Random Forest (with SMOTE):\n", classification_report(y_test, y_pred))


In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Apply SMOTE using the preprocessed X_transformed data
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_transformed, y)

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42
)

# Encode target labels to numerical values
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=12,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# Fit the model with the encoded target variables
xgb.fit(X_train, y_train_encoded)
y_pred_xgb_encoded = xgb.predict(X_test)

# Decode predictions back to original labels for classification report
y_pred_xgb = le.inverse_transform(y_pred_xgb_encoded)

print("XGBoost Results:\n", classification_report(y_test, y_pred_xgb))

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder # Import LabelEncoder

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)

# Encode target labels to numerical values for XGBoost
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)


xgb_model = XGBClassifier(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=7,
    scale_pos_weight=9 # Handle imbalance
)

# Fit the model with the encoded target variables
xgb.fit(X_train, y_train_encoded)
y_pred_xgb_encoded = xgb.predict(X_test)

# Decode predictions back to original labels for classification report
y_pred_xgb = le.inverse_transform(y_pred_xgb_encoded)

print("XGBoost Results:\n", classification_report(y_test, y_pred_xgb))

## 7. Model Training

Train multiple models and compare their performance.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print("Random Forest with SMOTE:\n", classification_report(y_test, y_pred_rf))


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Use engineered + scaled + encoded features directly (no PCA)
X_features = preprocessor.fit_transform(X)  # preprocessor from earlier steps
y_target = y

# Apply SMOTE for balance
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_features, y_target)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42
)

# Random Forest with tuned parameters
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print("Random Forest (with SMOTE):\n", classification_report(y_test, y_pred))


In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Apply SMOTE using the preprocessed X_transformed data
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_transformed, y)

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42
)

# Encode target labels to numerical values
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=12,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# Fit the model with the encoded target variables
xgb.fit(X_train, y_train_encoded)
y_pred_xgb_encoded = xgb.predict(X_test)

# Decode predictions back to original labels for classification report
y_pred_xgb = le.inverse_transform(y_pred_xgb_encoded)

print("XGBoost Results:\n", classification_report(y_test, y_pred_xgb))

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder # Import LabelEncoder

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)

# Encode target labels to numerical values for XGBoost
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)


xgb_model = XGBClassifier(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=7,
    scale_pos_weight=9 # Handle imbalance
)

# Fit the model with the encoded target variables
xgb.fit(X_train, y_train_encoded)
y_pred_xgb_encoded = xgb.predict(X_test)

# Decode predictions back to original labels for classification report
y_pred_xgb = le.inverse_transform(y_pred_xgb_encoded)

print("XGBoost Results:\n", classification_report(y_test, y_pred_xgb))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print("Random Forest with SMOTE:\n", classification_report(y_test, y_pred_rf))


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Use engineered + scaled + encoded features directly (no PCA)
X_features = preprocessor.fit_transform(X)  # preprocessor from earlier steps
y_target = y

# Apply SMOTE for balance
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_features, y_target)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42
)

# Random Forest with tuned parameters
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print("Random Forest (with SMOTE):\n", classification_report(y_test, y_pred))


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

param_dist = {
    'n_estimators': [200, 300, 500],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

rf = RandomForestClassifier(class_weight='balanced', random_state=42)

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=20,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train, y_train)
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy:", random_search.best_score_)


In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Apply SMOTE using the preprocessed X_transformed data
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_transformed, y)

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42
)

# Encode target labels to numerical values
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=12,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# Fit the model with the encoded target variables
xgb.fit(X_train, y_train_encoded)
y_pred_xgb_encoded = xgb.predict(X_test)

# Decode predictions back to original labels for classification report
y_pred_xgb = le.inverse_transform(y_pred_xgb_encoded)

print("XGBoost Results:\n", classification_report(y_test, y_pred_xgb))

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder # Import LabelEncoder

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)

# Encode target labels to numerical values for XGBoost
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)


xgb_model = XGBClassifier(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=7,
    scale_pos_weight=9 # Handle imbalance
)

# Fit the model with the encoded target variables
xgb.fit(X_train, y_train_encoded)
y_pred_xgb_encoded = xgb.predict(X_test)

# Decode predictions back to original labels for classification report
y_pred_xgb = le.inverse_transform(y_pred_xgb_encoded)

print("XGBoost Results:\n", classification_report(y_test, y_pred_xgb))

In [ ]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    min_data_in_leaf=10,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)


lgbm.fit(X_train, y_train)
y_pred_lgbm = lgbm.predict(X_test)

print("LightGBM Results:\n", classification_report(le.inverse_transform(y_test),
                                                  le.inverse_transform(y_pred_lgbm)))


In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier

# Train LightGBM
lgbm = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
lgbm.fit(X_train, y_train)

# Stacking ensemble
stack_model = StackingClassifier(
    estimators=[('rf', best_rf), ('xgb', xgb), ('lgbm', lgbm)],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5
)

stack_model.fit(X_train, y_train)
y_pred_stack = stack_model.predict(X_test)

print("Stacking Ensemble Results:\n", classification_report(le.inverse_transform(y_test),
                                                           le.inverse_transform(y_pred_stack)))


In [ ]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

lgbm.fit(X_train, y_train)
y_pred_lgbm = lgbm.predict(X_test)

print("LightGBM Results:\n", classification_report(le.inverse_transform(y_test),
                                                   le.inverse_transform(y_pred_lgbm)))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
# Assuming le (LabelEncoder), X_train, X_test, y_train, y_test, y_train_encoded, y_test_encoded are available globally.
# Assuming xgb (trained XGBoost model) is available globally and trained on y_train_encoded.

# Collect classification reports for each model
reports = {}

# 1. Re-initialize and re-train best_rf with its best parameters
# Best parameters from QI7EcV-f49Le output: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None}
# It MUST be trained on y_train_encoded for consistency with stacking and xgb.
best_rf = RandomForestClassifier(
    n_estimators=300,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='log2',
    max_depth=None,
    class_weight='balanced',
    random_state=42
)
best_rf.fit(X_train, y_train_encoded) # Train on the current X_train with 144 features and encoded y

y_pred_rf_encoded = best_rf.predict(X_test)
reports['RandomForest'] = classification_report(le.inverse_transform(y_test_encoded), # Use y_test_encoded for comparison consistency
                                                le.inverse_transform(y_pred_rf_encoded),
                                                output_dict=True)

# 2. Re-initialize and re-train lgbm with its parameters
# It MUST be trained on y_train_encoded for consistency with stacking and xgb.
lgbm = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    min_data_in_leaf=10,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
lgbm.fit(X_train, y_train_encoded) # Train on the current X_train with 144 features and encoded y

y_pred_lgbm_encoded = lgbm.predict(X_test)
reports['LightGBM'] = classification_report(le.inverse_transform(y_test_encoded), # Use y_test_encoded for comparison consistency
                                            le.inverse_transform(y_pred_lgbm_encoded),
                                            output_dict=True)

# 3. XGBoost model (assuming 'xgb' is already trained on y_train_encoded from e0RngTr5AP_l)
# The `xgb` variable exists and is trained on 144 features and encoded labels.
y_pred_xgb_encoded = xgb.predict(X_test) # xgb.predict directly returns encoded labels

reports['XGBoost'] = classification_report(le.inverse_transform(y_test_encoded), # Use y_test_encoded for comparison consistency
                                           le.inverse_transform(y_pred_xgb_encoded),
                                           output_dict=True)

# 4. Stacking ensemble
# All base estimators are now trained on encoded labels.
stack_model = StackingClassifier(
    estimators=[('rf', best_rf), ('xgb', xgb), ('lgbm', lgbm)],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5
)

# Fit stacking model with encoded labels
stack_model.fit(X_train, y_train_encoded)
y_pred_stack_encoded = stack_model.predict(X_test)

reports['StackingEnsemble'] = classification_report(le.inverse_transform(y_test_encoded), # Use y_test_encoded for comparison consistency
                                                    le.inverse_transform(y_pred_stack_encoded),
                                                    output_dict=True)

# Convert to DataFrame for comparison
df_list = []
for model_name, report in reports.items():
    df_report = pd.DataFrame(report).transpose()
    df_report['model'] = model_name
    df_list.append(df_report)

comparison_df = pd.concat(df_list)

# Extract overall metrics (accuracy, macro avg, weighted avg)
overall_metrics = comparison_df.loc[['accuracy','macro avg','weighted avg']]
overall_metrics = overall_metrics.reset_index().rename(columns={'index':'metric'})

# Pivot for plotting
pivot_df = overall_metrics.pivot(index='metric', columns='model', values='f1-score')

# Plot comparison
plt.figure(figsize=(10,6))
sns.barplot(data=pivot_df.T)
plt.title("Model Comparison (F1-scores for overall metrics)")
plt.ylabel("F1-score")
plt.xlabel("Model")
plt.legend(title="Metric")
plt.show()

# Print table for clarity
print("Overall Comparison:\n")
print(pivot_df)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print("Random Forest with SMOTE:\n", classification_report(y_test, y_pred_rf))


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Use engineered + scaled + encoded features directly (no PCA)
X_features = preprocessor.fit_transform(X)  # preprocessor from earlier steps
y_target = y

# Apply SMOTE for balance
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_features, y_target)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42
)

# Random Forest with tuned parameters
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print("Random Forest (with SMOTE):\n", classification_report(y_test, y_pred))


In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Apply SMOTE using the preprocessed X_transformed data
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_transformed, y)

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42
)

# Encode target labels to numerical values
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=12,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# Fit the model with the encoded target variables
xgb.fit(X_train, y_train_encoded)
y_pred_xgb_encoded = xgb.predict(X_test)

# Decode predictions back to original labels for classification report
y_pred_xgb = le.inverse_transform(y_pred_xgb_encoded)

print("XGBoost Results:\n", classification_report(y_test, y_pred_xgb))

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder # Import LabelEncoder

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)

# Encode target labels to numerical values for XGBoost
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)


xgb_model = XGBClassifier(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=7,
    scale_pos_weight=9 # Handle imbalance
)

# Fit the model with the encoded target variables
xgb.fit(X_train, y_train_encoded)
y_pred_xgb_encoded = xgb.predict(X_test)

# Decode predictions back to original labels for classification report
y_pred_xgb = le.inverse_transform(y_pred_xgb_encoded)

print("XGBoost Results:\n", classification_report(y_test, y_pred_xgb))

In [ ]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    min_data_in_leaf=10,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)


lgbm.fit(X_train, y_train)
y_pred_lgbm = lgbm.predict(X_test)

print("LightGBM Results:\n", classification_report(le.inverse_transform(y_test),
                                                  le.inverse_transform(y_pred_lgbm)))


In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier

# Train LightGBM
lgbm = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
lgbm.fit(X_train, y_train)

# Stacking ensemble
stack_model = StackingClassifier(
    estimators=[('rf', best_rf), ('xgb', xgb), ('lgbm', lgbm)],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5
)

stack_model.fit(X_train, y_train)
y_pred_stack = stack_model.predict(X_test)

print("Stacking Ensemble Results:\n", classification_report(le.inverse_transform(y_test),
                                                           le.inverse_transform(y_pred_stack)))


In [ ]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

lgbm.fit(X_train, y_train)
y_pred_lgbm = lgbm.predict(X_test)

print("LightGBM Results:\n", classification_report(le.inverse_transform(y_test),
                                                   le.inverse_transform(y_pred_lgbm)))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
# Assuming le (LabelEncoder), X_train, X_test, y_train, y_test, y_train_encoded, y_test_encoded are available globally.
# Assuming xgb (trained XGBoost model) is available globally and trained on y_train_encoded.

# Collect classification reports for each model
reports = {}

# 1. Re-initialize and re-train best_rf with its best parameters
# Best parameters from QI7EcV-f49Le output: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None}
# It MUST be trained on y_train_encoded for consistency with stacking and xgb.
best_rf = RandomForestClassifier(
    n_estimators=300,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='log2',
    max_depth=None,
    class_weight='balanced',
    random_state=42
)
best_rf.fit(X_train, y_train_encoded) # Train on the current X_train with 144 features and encoded y

y_pred_rf_encoded = best_rf.predict(X_test)
reports['RandomForest'] = classification_report(le.inverse_transform(y_test_encoded), # Use y_test_encoded for comparison consistency
                                                le.inverse_transform(y_pred_rf_encoded),
                                                output_dict=True)

# 2. Re-initialize and re-train lgbm with its parameters
# It MUST be trained on y_train_encoded for consistency with stacking and xgb.
lgbm = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    min_data_in_leaf=10,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
lgbm.fit(X_train, y_train_encoded) # Train on the current X_train with 144 features and encoded y

y_pred_lgbm_encoded = lgbm.predict(X_test)
reports['LightGBM'] = classification_report(le.inverse_transform(y_test_encoded), # Use y_test_encoded for comparison consistency
                                            le.inverse_transform(y_pred_lgbm_encoded),
                                            output_dict=True)

# 3. XGBoost model (assuming 'xgb' is already trained on y_train_encoded from e0RngTr5AP_l)
# The `xgb` variable exists and is trained on 144 features and encoded labels.
y_pred_xgb_encoded = xgb.predict(X_test) # xgb.predict directly returns encoded labels

reports['XGBoost'] = classification_report(le.inverse_transform(y_test_encoded), # Use y_test_encoded for comparison consistency
                                           le.inverse_transform(y_pred_xgb_encoded),
                                           output_dict=True)

# 4. Stacking ensemble
# All base estimators are now trained on encoded labels.
stack_model = StackingClassifier(
    estimators=[('rf', best_rf), ('xgb', xgb), ('lgbm', lgbm)],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5
)

# Fit stacking model with encoded labels
stack_model.fit(X_train, y_train_encoded)
y_pred_stack_encoded = stack_model.predict(X_test)

reports['StackingEnsemble'] = classification_report(le.inverse_transform(y_test_encoded), # Use y_test_encoded for comparison consistency
                                                    le.inverse_transform(y_pred_stack_encoded),
                                                    output_dict=True)

# Convert to DataFrame for comparison
df_list = []
for model_name, report in reports.items():
    df_report = pd.DataFrame(report).transpose()
    df_report['model'] = model_name
    df_list.append(df_report)

comparison_df = pd.concat(df_list)

# Extract overall metrics (accuracy, macro avg, weighted avg)
overall_metrics = comparison_df.loc[['accuracy','macro avg','weighted avg']]
overall_metrics = overall_metrics.reset_index().rename(columns={'index':'metric'})

# Pivot for plotting
pivot_df = overall_metrics.pivot(index='metric', columns='model', values='f1-score')

# Plot comparison
plt.figure(figsize=(10,6))
sns.barplot(data=pivot_df.T)
plt.title("Model Comparison (F1-scores for overall metrics)")
plt.ylabel("F1-score")
plt.xlabel("Model")
plt.legend(title="Metric")
plt.show()

# Print table for clarity
print("Overall Comparison:\n")
print(pivot_df)

## 8. Model Evaluation

Evaluate model performance using classification metrics.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib

from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print("Random Forest with SMOTE:\n", classification_report(y_test, y_pred_rf))


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Use engineered + scaled + encoded features directly (no PCA)
X_features = preprocessor.fit_transform(X)  # preprocessor from earlier steps
y_target = y

# Apply SMOTE for balance
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_features, y_target)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42
)

# Random Forest with tuned parameters
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print("Random Forest (with SMOTE):\n", classification_report(y_test, y_pred))


In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(best_rf, X_resampled, y_resampled, cv=5, scoring='accuracy')

print("Cross-validation scores:", cv_scores)
print("Mean CV accuracy:", cv_scores.mean())


In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Apply SMOTE using the preprocessed X_transformed data
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_transformed, y)

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42
)

# Encode target labels to numerical values
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=12,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# Fit the model with the encoded target variables
xgb.fit(X_train, y_train_encoded)
y_pred_xgb_encoded = xgb.predict(X_test)

# Decode predictions back to original labels for classification report
y_pred_xgb = le.inverse_transform(y_pred_xgb_encoded)

print("XGBoost Results:\n", classification_report(y_test, y_pred_xgb))

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder # Import LabelEncoder

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)

# Encode target labels to numerical values for XGBoost
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)


xgb_model = XGBClassifier(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=7,
    scale_pos_weight=9 # Handle imbalance
)

# Fit the model with the encoded target variables
xgb.fit(X_train, y_train_encoded)
y_pred_xgb_encoded = xgb.predict(X_test)

# Decode predictions back to original labels for classification report
y_pred_xgb = le.inverse_transform(y_pred_xgb_encoded)

print("XGBoost Results:\n", classification_report(y_test, y_pred_xgb))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print("Random Forest with SMOTE:\n", classification_report(y_test, y_pred_rf))


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Use engineered + scaled + encoded features directly (no PCA)
X_features = preprocessor.fit_transform(X)  # preprocessor from earlier steps
y_target = y

# Apply SMOTE for balance
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_features, y_target)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42
)

# Random Forest with tuned parameters
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print("Random Forest (with SMOTE):\n", classification_report(y_test, y_pred))


In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Apply SMOTE using the preprocessed X_transformed data
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_transformed, y)

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42
)

# Encode target labels to numerical values
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=12,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# Fit the model with the encoded target variables
xgb.fit(X_train, y_train_encoded)
y_pred_xgb_encoded = xgb.predict(X_test)

# Decode predictions back to original labels for classification report
y_pred_xgb = le.inverse_transform(y_pred_xgb_encoded)

print("XGBoost Results:\n", classification_report(y_test, y_pred_xgb))

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder # Import LabelEncoder

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)

# Encode target labels to numerical values for XGBoost
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)


xgb_model = XGBClassifier(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=7,
    scale_pos_weight=9 # Handle imbalance
)

# Fit the model with the encoded target variables
xgb.fit(X_train, y_train_encoded)
y_pred_xgb_encoded = xgb.predict(X_test)

# Decode predictions back to original labels for classification report
y_pred_xgb = le.inverse_transform(y_pred_xgb_encoded)

print("XGBoost Results:\n", classification_report(y_test, y_pred_xgb))

In [ ]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    min_data_in_leaf=10,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)


lgbm.fit(X_train, y_train)
y_pred_lgbm = lgbm.predict(X_test)

print("LightGBM Results:\n", classification_report(le.inverse_transform(y_test),
                                                  le.inverse_transform(y_pred_lgbm)))


In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier

# Train LightGBM
lgbm = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
lgbm.fit(X_train, y_train)

# Stacking ensemble
stack_model = StackingClassifier(
    estimators=[('rf', best_rf), ('xgb', xgb), ('lgbm', lgbm)],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5
)

stack_model.fit(X_train, y_train)
y_pred_stack = stack_model.predict(X_test)

print("Stacking Ensemble Results:\n", classification_report(le.inverse_transform(y_test),
                                                           le.inverse_transform(y_pred_stack)))


In [ ]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

lgbm.fit(X_train, y_train)
y_pred_lgbm = lgbm.predict(X_test)

print("LightGBM Results:\n", classification_report(le.inverse_transform(y_test),
                                                   le.inverse_transform(y_pred_lgbm)))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
# Assuming le (LabelEncoder), X_train, X_test, y_train, y_test, y_train_encoded, y_test_encoded are available globally.
# Assuming xgb (trained XGBoost model) is available globally and trained on y_train_encoded.

# Collect classification reports for each model
reports = {}

# 1. Re-initialize and re-train best_rf with its best parameters
# Best parameters from QI7EcV-f49Le output: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None}
# It MUST be trained on y_train_encoded for consistency with stacking and xgb.
best_rf = RandomForestClassifier(
    n_estimators=300,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='log2',
    max_depth=None,
    class_weight='balanced',
    random_state=42
)
best_rf.fit(X_train, y_train_encoded) # Train on the current X_train with 144 features and encoded y

y_pred_rf_encoded = best_rf.predict(X_test)
reports['RandomForest'] = classification_report(le.inverse_transform(y_test_encoded), # Use y_test_encoded for comparison consistency
                                                le.inverse_transform(y_pred_rf_encoded),
                                                output_dict=True)

# 2. Re-initialize and re-train lgbm with its parameters
# It MUST be trained on y_train_encoded for consistency with stacking and xgb.
lgbm = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    min_data_in_leaf=10,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
lgbm.fit(X_train, y_train_encoded) # Train on the current X_train with 144 features and encoded y

y_pred_lgbm_encoded = lgbm.predict(X_test)
reports['LightGBM'] = classification_report(le.inverse_transform(y_test_encoded), # Use y_test_encoded for comparison consistency
                                            le.inverse_transform(y_pred_lgbm_encoded),
                                            output_dict=True)

# 3. XGBoost model (assuming 'xgb' is already trained on y_train_encoded from e0RngTr5AP_l)
# The `xgb` variable exists and is trained on 144 features and encoded labels.
y_pred_xgb_encoded = xgb.predict(X_test) # xgb.predict directly returns encoded labels

reports['XGBoost'] = classification_report(le.inverse_transform(y_test_encoded), # Use y_test_encoded for comparison consistency
                                           le.inverse_transform(y_pred_xgb_encoded),
                                           output_dict=True)

# 4. Stacking ensemble
# All base estimators are now trained on encoded labels.
stack_model = StackingClassifier(
    estimators=[('rf', best_rf), ('xgb', xgb), ('lgbm', lgbm)],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5
)

# Fit stacking model with encoded labels
stack_model.fit(X_train, y_train_encoded)
y_pred_stack_encoded = stack_model.predict(X_test)

reports['StackingEnsemble'] = classification_report(le.inverse_transform(y_test_encoded), # Use y_test_encoded for comparison consistency
                                                    le.inverse_transform(y_pred_stack_encoded),
                                                    output_dict=True)

# Convert to DataFrame for comparison
df_list = []
for model_name, report in reports.items():
    df_report = pd.DataFrame(report).transpose()
    df_report['model'] = model_name
    df_list.append(df_report)

comparison_df = pd.concat(df_list)

# Extract overall metrics (accuracy, macro avg, weighted avg)
overall_metrics = comparison_df.loc[['accuracy','macro avg','weighted avg']]
overall_metrics = overall_metrics.reset_index().rename(columns={'index':'metric'})

# Pivot for plotting
pivot_df = overall_metrics.pivot(index='metric', columns='model', values='f1-score')

# Plot comparison
plt.figure(figsize=(10,6))
sns.barplot(data=pivot_df.T)
plt.title("Model Comparison (F1-scores for overall metrics)")
plt.ylabel("F1-score")
plt.xlabel("Model")
plt.legend(title="Metric")
plt.show()

# Print table for clarity
print("Overall Comparison:\n")
print(pivot_df)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print("Random Forest with SMOTE:\n", classification_report(y_test, y_pred_rf))


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Use engineered + scaled + encoded features directly (no PCA)
X_features = preprocessor.fit_transform(X)  # preprocessor from earlier steps
y_target = y

# Apply SMOTE for balance
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_features, y_target)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42
)

# Random Forest with tuned parameters
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print("Random Forest (with SMOTE):\n", classification_report(y_test, y_pred))


In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Apply SMOTE using the preprocessed X_transformed data
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_transformed, y)

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42
)

# Encode target labels to numerical values
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=12,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# Fit the model with the encoded target variables
xgb.fit(X_train, y_train_encoded)
y_pred_xgb_encoded = xgb.predict(X_test)

# Decode predictions back to original labels for classification report
y_pred_xgb = le.inverse_transform(y_pred_xgb_encoded)

print("XGBoost Results:\n", classification_report(y_test, y_pred_xgb))

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder # Import LabelEncoder

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)

# Encode target labels to numerical values for XGBoost
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)


xgb_model = XGBClassifier(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=7,
    scale_pos_weight=9 # Handle imbalance
)

# Fit the model with the encoded target variables
xgb.fit(X_train, y_train_encoded)
y_pred_xgb_encoded = xgb.predict(X_test)

# Decode predictions back to original labels for classification report
y_pred_xgb = le.inverse_transform(y_pred_xgb_encoded)

print("XGBoost Results:\n", classification_report(y_test, y_pred_xgb))

In [ ]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    min_data_in_leaf=10,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)


lgbm.fit(X_train, y_train)
y_pred_lgbm = lgbm.predict(X_test)

print("LightGBM Results:\n", classification_report(le.inverse_transform(y_test),
                                                  le.inverse_transform(y_pred_lgbm)))


In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier

# Train LightGBM
lgbm = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
lgbm.fit(X_train, y_train)

# Stacking ensemble
stack_model = StackingClassifier(
    estimators=[('rf', best_rf), ('xgb', xgb), ('lgbm', lgbm)],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5
)

stack_model.fit(X_train, y_train)
y_pred_stack = stack_model.predict(X_test)

print("Stacking Ensemble Results:\n", classification_report(le.inverse_transform(y_test),
                                                           le.inverse_transform(y_pred_stack)))


In [ ]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

lgbm.fit(X_train, y_train)
y_pred_lgbm = lgbm.predict(X_test)

print("LightGBM Results:\n", classification_report(le.inverse_transform(y_test),
                                                   le.inverse_transform(y_pred_lgbm)))


In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

stack_model = StackingClassifier(
    estimators=[('rf', best_rf), ('xgb', xgb), ('lgbm', lgbm)],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5
)

stack_model.fit(X_train, y_train)
y_pred_stack = stack_model.predict(X_test)

print("Stacking Ensemble Results:\n", classification_report(le.inverse_transform(y_test),
                                                           le.inverse_transform(y_pred_stack)))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
# Assuming le (LabelEncoder), X_train, X_test, y_train, y_test, y_train_encoded, y_test_encoded are available globally.
# Assuming xgb (trained XGBoost model) is available globally and trained on y_train_encoded.

# Collect classification reports for each model
reports = {}

# 1. Re-initialize and re-train best_rf with its best parameters
# Best parameters from QI7EcV-f49Le output: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None}
# It MUST be trained on y_train_encoded for consistency with stacking and xgb.
best_rf = RandomForestClassifier(
    n_estimators=300,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='log2',
    max_depth=None,
    class_weight='balanced',
    random_state=42
)
best_rf.fit(X_train, y_train_encoded) # Train on the current X_train with 144 features and encoded y

y_pred_rf_encoded = best_rf.predict(X_test)
reports['RandomForest'] = classification_report(le.inverse_transform(y_test_encoded), # Use y_test_encoded for comparison consistency
                                                le.inverse_transform(y_pred_rf_encoded),
                                                output_dict=True)

# 2. Re-initialize and re-train lgbm with its parameters
# It MUST be trained on y_train_encoded for consistency with stacking and xgb.
lgbm = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    min_data_in_leaf=10,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
lgbm.fit(X_train, y_train_encoded) # Train on the current X_train with 144 features and encoded y

y_pred_lgbm_encoded = lgbm.predict(X_test)
reports['LightGBM'] = classification_report(le.inverse_transform(y_test_encoded), # Use y_test_encoded for comparison consistency
                                            le.inverse_transform(y_pred_lgbm_encoded),
                                            output_dict=True)

# 3. XGBoost model (assuming 'xgb' is already trained on y_train_encoded from e0RngTr5AP_l)
# The `xgb` variable exists and is trained on 144 features and encoded labels.
y_pred_xgb_encoded = xgb.predict(X_test) # xgb.predict directly returns encoded labels

reports['XGBoost'] = classification_report(le.inverse_transform(y_test_encoded), # Use y_test_encoded for comparison consistency
                                           le.inverse_transform(y_pred_xgb_encoded),
                                           output_dict=True)

# 4. Stacking ensemble
# All base estimators are now trained on encoded labels.
stack_model = StackingClassifier(
    estimators=[('rf', best_rf), ('xgb', xgb), ('lgbm', lgbm)],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5
)

# Fit stacking model with encoded labels
stack_model.fit(X_train, y_train_encoded)
y_pred_stack_encoded = stack_model.predict(X_test)

reports['StackingEnsemble'] = classification_report(le.inverse_transform(y_test_encoded), # Use y_test_encoded for comparison consistency
                                                    le.inverse_transform(y_pred_stack_encoded),
                                                    output_dict=True)

# Convert to DataFrame for comparison
df_list = []
for model_name, report in reports.items():
    df_report = pd.DataFrame(report).transpose()
    df_report['model'] = model_name
    df_list.append(df_report)

comparison_df = pd.concat(df_list)

# Extract overall metrics (accuracy, macro avg, weighted avg)
overall_metrics = comparison_df.loc[['accuracy','macro avg','weighted avg']]
overall_metrics = overall_metrics.reset_index().rename(columns={'index':'metric'})

# Pivot for plotting
pivot_df = overall_metrics.pivot(index='metric', columns='model', values='f1-score')

# Plot comparison
plt.figure(figsize=(10,6))
sns.barplot(data=pivot_df.T)
plt.title("Model Comparison (F1-scores for overall metrics)")
plt.ylabel("F1-score")
plt.xlabel("Model")
plt.legend(title="Metric")
plt.show()

# Print table for clarity
print("Overall Comparison:\n")
print(pivot_df)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report

# Collect classification reports for each model
reports = {}
reports['RandomForest'] = classification_report(le.inverse_transform(y_test_encoded),
                                                le.inverse_transform(best_rf.predict(X_test)),
                                                output_dict=True)
reports['XGBoost'] = classification_report(le.inverse_transform(y_test_encoded),
                                           le.inverse_transform(xgb.predict(X_test)),
                                           output_dict=True)
reports['LightGBM'] = classification_report(le.inverse_transform(y_test_encoded),
                                            le.inverse_transform(lgbm.predict(X_test)),
                                            output_dict=True)
reports['StackingEnsemble'] = classification_report(le.inverse_transform(y_test_encoded),
                                                    le.inverse_transform(stack_model.predict(X_test)),
                                                    output_dict=True)

# Convert reports into a DataFrame for overall metrics (as done previously)
df_list_overall = []
for model_name, report in reports.items():
    df_report = pd.DataFrame(report).transpose() # Index has class names + overall metrics
    df_report['model'] = model_name
    df_list_overall.append(df_report)

comparison_df = pd.concat(df_list_overall)

# Extract overall metrics (accuracy, macro avg, weighted avg)
overall_metrics = comparison_df.loc[['accuracy','macro avg','weighted avg']]
overall_metrics = overall_metrics.reset_index().rename(columns={'index':'metric'})

# Pivot for plotting overall metrics
pivot_df = overall_metrics.pivot(index='metric', columns='model', values='f1-score')

# Plot comparison for overall metrics
plt.figure(figsize=(10,6))
sns.barplot(data=pivot_df.T)
plt.title("Model Comparison (F1-scores for overall metrics)")
plt.ylabel("F1-score")
plt.xlabel("Model")
plt.legend(title="Metric")
plt.show()

# Print table for clarity
print("Overall Comparison:\n")
print(pivot_df)

# --- Corrected section for per-class metrics ---

# Define metrics to extract
metrics = ['precision','recall','f1-score']

# Create a list to store per-class dataframes
df_list_per_class = []

for model_name, report in reports.items():
    # Filter the report dictionary to include only class-specific metrics
    # The keys for classes are numerical (0, 1, 2, 3) in the report dictionaries from the kernel state
    class_labels_in_report = [k for k in report.keys() if k.isdigit()] # Get numerical keys as strings '0', '1', '2', '3'
    class_report_filtered = {cls_key: report[cls_key] for cls_key in class_labels_in_report}

    # Create DataFrame for current model's per-class metrics
    df_per_class = pd.DataFrame(class_report_filtered).transpose()
    df_per_class['model'] = model_name
    df_list_per_class.append(df_per_class)

# Concatenate all per-class dataframes
per_class_df = pd.concat(df_list_per_class)

# Reset index (which are the numerical class labels as strings) and melt for plotting
per_class_long = per_class_df.reset_index().rename(columns={'index':'encoded_class'}).melt(id_vars=['encoded_class','model'],
                                 value_vars=metrics,
                                 var_name='metric',
                                 value_name='score')

# Map the encoded_class back to original string class names for better visualization
# Need to convert 'encoded_class' column to int first, then map using LabelEncoder
per_class_long['class'] = per_class_long['encoded_class'].astype(int).map(lambda x: le.inverse_transform([x])[0])

# Plot per-class comparison
plt.figure(figsize=(14,8))
sns.barplot(data=per_class_long, x='class', y='score', hue='model')
plt.title("Per-Class Model Comparison (Precision, Recall, F1)")
plt.ylabel("Score")
plt.xlabel("Class")
plt.legend(title="Model")
plt.show()

# Print table for clarity
print("Per-Class Comparison:\n")
# To print per_class_df with original class names, we'll map its index
per_class_df_to_print = per_class_df.copy()
per_class_df_to_print.index = per_class_df_to_print.index.astype(int).map(lambda x: le.inverse_transform([x])[0])
print(per_class_df_to_print)

## 9. Save Trained Model



In [24]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


# Re-identify numerical and categorical features from the current state of X
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

# Define the preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'
)
print("Preprocessor defined.")

# Encode the target variable `y` once before resampling
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print("Target variable (y) encoded with LabelEncoder (le).")

# Apply preprocessing to X and then SMOTE
X_transformed = preprocessor.fit_transform(X)
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_transformed, y_encoded)
print(f"Data preprocessed, resampled with SMOTE. X_resampled shape: {X_resampled.shape}")

# Split the resampled data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)
print(f"Train/Test split complete. X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

# Train RandomForest (best_rf)
best_rf = RandomForestClassifier(
    n_estimators=300,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='log2',
    max_depth=None,
    class_weight='balanced',
    random_state=42
)
best_rf.fit(X_train, y_train)
print("RandomForest model (best_rf) trained.")

# Train XGBoost
xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=12,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
xgb.fit(X_train, y_train)
print("XGBoost model (xgb) trained.")

# Train LightGBM
lgbm = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=20,
    min_data_in_leaf=10,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
lgbm.fit(X_train, y_train)
print("LightGBM model (lgbm) trained.")

# Train Stacking Ensemble
stack_model = StackingClassifier(
    estimators=[('rf', best_rf), ('xgb', xgb), ('lgbm', lgbm)],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5
)
stack_model.fit(X_train, y_train)
print("Stacking Ensemble model (stack_model) trained.")

print("All required models and preprocessor defined and trained.")

Preprocessor defined.
Target variable (y) encoded with LabelEncoder (le).
Data preprocessed, resampled with SMOTE. X_resampled shape: (5584, 119)
Train/Test split complete. X_train shape: (4467, 119), y_train shape: (4467,)
RandomForest model (best_rf) trained.
XGBoost model (xgb) trained.
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009523 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 25938
[LightGBM] [Info] Number of data points in the train set: 4467, number of used features: 119
[LightGBM] [Info] Start training from score -1.386071
[LightGBM] [Info] Start training from score -1.386071
[Li

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008634 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 23219
[LightGBM] [Info] Number of data points in the train set: 3573, number of used features: 119
[LightGBM] [Info] Start training from score -1.386574
[LightGBM] [Info] Start training from score -1.386574
[LightGBM] [Info] Start training from score -1.386574
[LightGBM] [Info] Start training from score -1.385455


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009025 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 23589
[LightGBM] [Info] Number of data points in the train set: 3574, number of used features: 119
[LightGBM] [Info] Start training from score -1.385735
[LightGBM] [Info] Start training from score -1.386854
[LightGBM] [Info] Start training from score -1.386854
[LightGBM] [Info] Start training from score -1.385735


...

[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009417 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 23006
[LightGBM] [Info] Number of data points in the train set: 3574, number of used features: 119
[LightGBM] [Info] Start training from score -1.385735
[LightGBM] [Info] Start training from score -1.385735
[LightGBM] [Info] Start training from score -1.386854
[LightGBM] [Info] Start training from score -1.386854


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.011915 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 23591
[LightGBM] [Info] Number of data points in the train set: 3574, number of used features: 119
[LightGBM] [Info] Start training from score -1.385735
[LightGBM] [Info] Start training from score -1.385735
[LightGBM] [Info] Start training from score -1.386854
[LightGBM] [Info] Start training from score -1.386854


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
Stacking Ensemble model (stack_model) trained.
All required models and preprocessor defined and trained.


In [25]:
from sklearn.metrics import classification_report

y_pred_stack = stack_model.predict(X_test)
print("Stacking Ensemble Results:\n", classification_report(le.inverse_transform(y_test), le.inverse_transform(y_pred_stack)))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
Stacking Ensemble Results:
               precision    recall  f1-score   support

    Critical       0.92      0.85      0.89       279
        High       0.77      0.74      0.75       279
         Low       0.89      0.89      0.89       280
      Medium       0.71      0.78      0.75       279

    accuracy                           0.82      1117
   macro avg       0.82      0.82      0.82      1117
weighted avg       0.82      0.82      0.82      1117



In [26]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline # Alias to avoid clash with sklearn.pipeline.Pipeline

# Ensure numerical_features and categorical_features are consistent
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

# Perform a new train/test split on the *raw* data for the end-to-end pipeline
# y_encoded is assumed to be available from previous cells
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Define the full end-to-end pipeline
pipeline = ImbPipeline(
    steps=[
        ("preprocessor", preprocessor), # Reusing the preprocessor defined earlier
        ("smote", SMOTE(random_state=42)), # Including SMOTE in the pipeline
        ("model", stack_model) # Reusing the trained stack_model
    ]
)

# Fit the full pipeline on the *raw* training data
pipeline.fit(X_train_raw, y_train_raw)

print("Full end-to-end pipeline including preprocessing, SMOTE, and Stacking Classifier has been fitted on raw data.")

[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012340 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 26428
[LightGBM] [Info] Number of data points in the train set: 4468, number of used features: 119
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_sampl

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008150 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 26300
[LightGBM] [Info] Number of data points in the train set: 3574, number of used features: 119
[LightGBM] [Info] Start training from score -1.386854
[LightGBM] [Info] Start training from score -1.385735
[LightGBM] [Info] Start training from score -1.385735
[LightGBM] [Info] Start training from score -1.386854


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008010 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 25866
[LightGBM] [Info] Number of data points in the train set: 3574, number of used features: 119
[LightGBM] [Info] Start training from score -1.386854
[LightGBM] [Info] Start training from score -1.385735
[LightGBM] [Info] Start training from score -1.386854
[LightGBM] [Info] Start training from score -1.385735


...

[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006850 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 16117
[LightGBM] [Info] Number of data points in the train set: 3575, number of used features: 119
[LightGBM] [Info] Start training from score -1.386015
[LightGBM] [Info] Start training from score -1.386015
[LightGBM] [Info] Start training from score -1.387134
[LightGBM] [Info] Start training from score -1.386015


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006810 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 11504
[LightGBM] [Info] Number of data points in the train set: 3575, number of used features: 119
[LightGBM] [Info] Start training from score -1.386015
[LightGBM] [Info] Start training from score -1.387134
[LightGBM] [Info] Start training from score -1.386015
[LightGBM] [Info] Start training from score -1.386015


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
Full end-to-end pipeline including preprocessing, SMOTE, and Stacking Classifier has been fitted on raw data.


In [27]:
import joblib
from google.colab import files
joblib.dump(stack_model, "stack_model.pkl")
files.download("stack_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
import pandas as pd # Ensure pandas is imported

pd.DataFrame(X_test).to_csv("X_test.csv", index=False)
pd.DataFrame(y_test, columns=['Risk_Level']).to_csv("y_test.csv", index=False)
files.download("X_test.csv")
files.download("y_test.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:
joblib.dump(pipeline, "full_pipeline.pkl")
files.download("full_pipeline.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>